### BP infection batch processing
Runs the infection segmentation and feature-extraction pipeline (from `1_SP_infection_image_viz_mp.ipynb`) on all images in `data_folder`. Each ND2 is looped over positions, MIP'd per FOV, and saved as one CSV per image (with a `position` column) under `results/experiment_id`. No filtering or visualization.

### Config: data folder, OCs, output directory

In [ ]:
import os
from pathlib import Path

from utils import list_images, check_marker_metadata_match, read_image

# Path to folder containing images
data_folder = r"Z:\Kai.beckwith\2026-09-07_KSB_iMACS_MTb_BSL2_4h-8d\2026-09-15_KSB_iMACS_MTb_BSL2_8d"

# Markers for intensity analysis: (channel_name, channel_nr), 0-based
MARKERS = [("SD_DAPI", 0), ("SD_BF", 1)]

# Define brightfield channel position to use as input for CellposeSAM-mediated cell segmentation
BRIGHTFIELD_CHANNEL = 1

# Define Mtb channel position to use as input for APOC ObjectSegmenter
MTB_CHANNEL = 0

images = list_images(data_folder, format="nd2")
experiment_id = Path(data_folder).name
out_dir = Path("results") / experiment_id
os.makedirs(out_dir, exist_ok=True)
print(f"Found {len(images)} images. Output: {out_dir}")

# Check if the defined marker position match the metadata in the image files
metadata_match = check_marker_metadata_match(images, MARKERS)
if not metadata_match:
    raise RuntimeError(
        "Marker/metadata mismatch detected in one or more files. "
        "Fix markers or input data before continuing."
    )

### Imports and model load (run once)

In [ ]:
from cellpose import models, core
from skimage.segmentation import clear_border
from skimage.measure import regionprops_table
import pandas as pd
import numpy as np
import apoc

from utils_infection import detect_infected_cells, detect_infection_load

import logging
logging.getLogger("tifffile").setLevel(logging.ERROR)

if core.use_gpu() == False:
    raise ImportError("No GPU access, change your runtime")

# Load fine-tuned Cellpose-SAM for infection brightfield segmentation
model = models.CellposeModel(
    gpu=True,
    pretrained_model="./models/CPSAM_shirik_ft",
)

# Load pretrained Object Classifier for Mycobacterium infection detection
mtb_cl_filename = "./models/Mtb_segmenter_kbeckwith.cl"
mtb_segmenter = apoc.ObjectSegmenter(opencl_filename=mtb_cl_filename)

### Morphology and intensity property lists

In [ ]:
morphology_properties = [
    "label",
    "area",
    "area_bbox",
    "area_convex",
    "area_filled",
    "axis_major_length",
    "axis_minor_length",
    "equivalent_diameter_area",
    "euler_number",
    "extent",
    "feret_diameter_max",
    "solidity",
    "inertia_tensor_eigvals",
]

intensity_properties = [
    "label",
    "intensity_mean",
    "intensity_min",
    "intensity_max",
    "intensity_std",
]

### Process all images and save CSVs

In [ ]:
infection_stats = []

for i, img_filepath in enumerate(images):
    print(f"Processing {i + 1}/{len(images)}: {Path(img_filepath).name}")

    img, filename = read_image(img_filepath, log=False)
    n_positions = img.shape[0]
    position_dfs = []

    for position in range(n_positions):
        print(f"  Position {position + 1}/{n_positions}")

        # Select FOV, then MIP over Z before feeding the image into the pipeline
        img_pos = np.max(img[position], axis=0)

        cell_labels, flows, styles = model.eval(img_pos[BRIGHTFIELD_CHANNEL], niter=1000)
        cell_labels = clear_border(cell_labels)

        mtb_labels, infected_cell_labels = detect_infected_cells(
            img_pos, mtb_segmenter, cell_labels, MTB_CHANNEL, filename, infection_stats,
            position=position,
        )

        props_morphology = regionprops_table(
            label_image=cell_labels,
            properties=morphology_properties,
        )
        props_df = pd.DataFrame(props_morphology)

        for marker_name, ch_nr in MARKERS:
            props = regionprops_table(
                label_image=cell_labels,
                intensity_image=img_pos[ch_nr],
                properties=intensity_properties,
            )
            intensity_df = pd.DataFrame(props)

            prefix = f"{marker_name}"
            rename_map = {"label": "label"}
            for prop in intensity_properties:
                if prop == "label":
                    continue
                if prop.startswith("intensity_"):
                    suffix = prop.replace("intensity_", "")
                    rename_map[prop] = f"{prefix}_{suffix}_int"
            intensity_df.rename(columns=rename_map, inplace=True)

            mean_col = rename_map["intensity_mean"]
            max_col = rename_map["intensity_max"]
            intensity_df[f"{prefix}_max_mean_ratio"] = (
                intensity_df[max_col] / intensity_df[mean_col].replace(0, np.nan)
            )

            props_df = props_df.merge(intensity_df, on="label")
            props_df[f"{prefix}_sum_int"] = props_df[mean_col] * props_df["area"]

        props_df = detect_infection_load(mtb_labels, cell_labels, props_df)

        col_idx = props_df.columns.get_loc("label")
        props_df.insert(col_idx + 1, "Mtb_infected_cell", props_df["label"].isin(infected_cell_labels))

        props_df.insert(0, "filename", filename)
        props_df.insert(1, "position", position)
        position_dfs.append(props_df)

    props_df = pd.concat(position_dfs, ignore_index=True)
    csv_path = out_dir / f"{filename}.csv"
    props_df.to_csv(csv_path, index=False)
    print(f"  -> {csv_path}")

infection_stats_df = pd.DataFrame(infection_stats)
infection_stats_path = out_dir / "infection_summary.csv"
infection_stats_df.to_csv(infection_stats_path, index=False)
print(f"Infection summary: {infection_stats_path}")
print("Done.")